# V-Scan Object Cutout
Crops a pinhole image of every annotated object out of a V-Scan equirectangular panorama, using the object's oriented bounding box (`main_bb.json`). Each crop is saved under `cropped_object_images/`, named after the box `id`.

This follows the same bounding-box-to-pinhole reprojection used for the office/real-scan data, adapted to the V-Scan box format (8 explicit corner points instead of centre/size/rotation).

In [ ]:
import os
import json
import numpy as np
import cv2
from PIL import Image
import open3d as o3d

import os
import sys
sys.path.insert(0, '../')
import drm
import drm.detect
import drm.generate
from pathlib import Path

# --- Paths ---
scene_dir = Path(r"/home/jvermandere/datasets/V-Scan/data/Industrial_1_Leica-P30_1775813780764")

SCAN_FILE = "main.txt"
PANO_FILE = "pano.png"
BB_FILE = "main_bb.json"

outDir = scene_dir / "cropped_object_images"
os.makedirs(outDir, exist_ok=True)
outDir

## File Loading
Load the bounding box annotations, the equirectangular panorama, and the scanner pose (`panomatrix`) for the scene.

In [ ]:
bbData = drm.detect.load_gt_boxes_raw(scene_dir / BB_FILE)
bbPoints = []
for box in bbData:
    bbPoints.append(box["points"])

trans_matrix = drm.read_transform_matrix(scene_dir / SCAN_FILE, apply_unity_conversion=True)
pano_image = np.array(Image.open(scene_dir / PANO_FILE).convert("RGB"))
annotated = drm.generate.draw_plane_corners_on_pano(o3d.geometry.PointCloud(points=o3d.utility.Vector3dVector(np.array(bbPoints).reshape((-1,3)))), pano_image, trans_matrix, radius=5)
masked_image = Image.fromarray(annotated)

masked_image

## Boundingbox Image Crop
Create pinhole camera cropouts of the pano image using each object's bounding box.

In [ ]:
def polygon_to_pinhole(equi_img, poly_pixels, out_res=(800, 600), margin=1.1):
    H, W, _ = equi_img.shape

    # Convert polygon vertices to spherical coordinates
    thetas = [(x / W) * 2 * np.pi - np.pi for x, y in poly_pixels]
    phis = [np.pi / 2 - (y / H) * np.pi for x, y in poly_pixels]

    # Convert spherical to 3D Cartesian
    verts_3d = np.array([
        [np.cos(phi) * np.sin(theta), np.sin(phi), np.cos(phi) * np.cos(theta)]
        for theta, phi in zip(thetas, phis)
    ])

    # Camera look direction = normalized mean of vertices
    center_dir = verts_3d.mean(axis=0)
    center_dir /= np.linalg.norm(center_dir)

    # Rotation matrix: rotate z-axis to center_dir
    def look_at_matrix(forward):
        forward = forward / np.linalg.norm(forward)
        tmp = np.array([0, 1, 0]) if abs(forward[1]) < 0.99 else np.array([1, 0, 0])
        right = np.cross(tmp, forward)
        right /= np.linalg.norm(right)
        up = np.cross(forward, right)
        return np.stack([right, up, forward], axis=1)

    R = look_at_matrix(center_dir)

    # Transform all vertices to camera frame
    verts_cam = verts_3d @ R
    x_max, x_min = verts_cam[:, 0].max(), verts_cam[:, 0].min()
    y_max, y_min = verts_cam[:, 1].max(), verts_cam[:, 1].min()
    z_max, z_min = verts_cam[:, 2].max(), verts_cam[:, 2].min()

    # Compute FOV to include all points, with margin
    fov_x = 2 * np.arctan(margin * max(abs(x_max), abs(x_min)) / max(z_max, 1e-6))
    fov_y = 2 * np.arctan(margin * max(abs(y_max), abs(y_min)) / max(z_max, 1e-6))

    # Use the larger FOV for both directions to keep aspect ratio
    fov = np.rad2deg(max(fov_x, fov_y))

    # Pinhole camera intrinsics
    w_out, h_out = out_res
    fx = fy = 0.5 * w_out / np.tan(np.deg2rad(fov) / 2)
    cx_out, cy_out = w_out / 2, h_out / 2

    # Pixel grid
    xx, yy = np.meshgrid(np.arange(w_out), np.arange(h_out))
    x = (xx - cx_out) / fx
    y = (cy_out - yy) / fy   # was (yy - cy_out) / fy
    z = np.ones_like(x)

    dirs = np.stack([x, y, z], axis=-1)
    dirs /= np.linalg.norm(dirs, axis=-1, keepdims=True)

    # Rotate directions to point at polygon center
    dirs = dirs @ R.T

    # Convert to spherical coordinates for sampling
    lon = np.arctan2(dirs[..., 0], dirs[..., 2])
    lat = np.arcsin(dirs[..., 1])

    # Map to equirectangular pixels
    u = ((lon + np.pi) / (2 * np.pi) * W).astype(np.float32)
    v = ((np.pi / 2 - lat) / np.pi * H).astype(np.float32)

    # Sample with OpenCV
    pinhole_img = cv2.remap(equi_img, u, v, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_WRAP)
    return pinhole_img

### Generate Cropped Object Images
Crop and save a pinhole image for every box, named after its `id`.

In [ ]:
def sanitize_filename(name):
    # box ids are GUID-like strings (e.g. "...(Clone)"); strip the few
    # characters that are unsafe as filenames while keeping the id readable
    return "".join(c if c not in '/\\:*?"<>|' else "_" for c in name)

out_res = (800, 600)
margin = 1.5
Ry = np.array([
    [ 0, 0, -1, 0],
    [ 0, 1,  0, 0],
    [ 1, 0,  0, 0],
    [ 0, 0,  0, 1]
], dtype=np.float64)
pano_h, pano_w = pano_image.shape[:2]

saved_files = []
for box in bbData:
    box_id = box["id"]
    pts3d = box["points"]
    tMat = Ry @ np.linalg.inv(trans_matrix)
    uv = drm.transform_xyz_to_uv(pts3d, tMat)     # (V, 2)
    pts = np.round(uv * [pano_w - 1, pano_h - 1]).astype(np.int32)

    crop = polygon_to_pinhole(pano_image, pts, out_res=out_res, margin=margin)

    filename = os.path.join(outDir, f"{sanitize_filename(box_id)}.png")
    cv2.imwrite(filename, cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
    saved_files.append(filename)

print(f"Saved {len(saved_files)} cropped object images to {outDir}")

### Preview

In [ ]:
Image.open(saved_files[0])

## Full scene Processing


In [ ]:
def process_scene(
    scene_dir: Path,
    *,
    scan_file: str = "main.txt",
    pano_file: str = "pano.png",
    bb_file: str = "main_bb.json",
    out_subdir: str = "cropped_object_images",
    out_res: tuple[int, int] = (800, 600),
    margin: float = 1.5,
) -> list[str]:
    """
    Run the object cutout pipeline for a single V-Scan scene folder.
    Returns the list of saved crop paths (empty if files/boxes are missing).
    """
    scene_dir = Path(scene_dir)
    bb_path, pano_path, scan_path = scene_dir / bb_file, scene_dir / pano_file, scene_dir / scan_file

    missing = [p.name for p in (bb_path, pano_path, scan_path) if not p.exists()]
    if missing:
        print(f"[skip] {scene_dir.name}: missing {missing}")
        return []

    bbData = drm.detect.load_gt_boxes_raw(bb_path)
    if not bbData:
        print(f"[skip] {scene_dir.name}: no boxes in {bb_file}")
        return []

    out_dir = scene_dir / out_subdir
    out_dir.mkdir(exist_ok=True)

    trans_matrix = drm.read_transform_matrix(scan_path, apply_unity_conversion=True)
    pano_image = np.array(Image.open(pano_path).convert("RGB"))
    pano_h, pano_w = pano_image.shape[:2]

    Ry = np.array([
        [0, 0, -1, 0],
        [0, 1,  0, 0],
        [1, 0,  0, 0],
        [0, 0,  0, 1],
    ], dtype=np.float64)
    tMat = Ry @ np.linalg.inv(trans_matrix)

    saved_files = []
    for box in bbData:
        uv = drm.transform_xyz_to_uv(box["points"], tMat)
        pts = np.round(uv * [pano_w - 1, pano_h - 1]).astype(np.int32)

        crop = polygon_to_pinhole(pano_image, pts, out_res=out_res, margin=margin)

        filename = out_dir / f"{sanitize_filename(box['id'])}.png"
        cv2.imwrite(str(filename), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        saved_files.append(str(filename))

    print(f"[ok] {scene_dir.name}: saved {len(saved_files)} crops to {out_dir}")
    return saved_files


def process_dataset(dataset_root: Path, **scene_kwargs) -> dict[str, list[str]]:
    """
    Runs process_scene() over every immediate subfolder of dataset_root.
    Folders missing the required files are skipped; folders that raise
    an exception are logged and skipped so one bad scene doesn't abort
    the rest of the batch.
    """
    dataset_root = Path(dataset_root)
    scene_dirs = sorted(p for p in dataset_root.iterdir() if p.is_dir())
    print(f"Found {len(scene_dirs)} scene folders under {dataset_root}")

    results = {}
    for scene_dir in scene_dirs:
        try:
            results[scene_dir.name] = process_scene(scene_dir, **scene_kwargs)
        except Exception as e:
            print(f"[error] {scene_dir.name}: {e}")
            results[scene_dir.name] = []

    total = sum(len(v) for v in results.values())
    n_ok = sum(1 for v in results.values() if v)
    print(f"\nDone: {total} crops saved across {n_ok}/{len(scene_dirs)} scenes")
    return results

dataset_root = Path("/home/jvermandere/datasets/V-Scan/data")
results = process_dataset(dataset_root)